#### This notebook generates a .bin file with map data for use by the Aggie XLMA (C++) application. 
Please generate the necessary file and update the array in state.h.
US shapefiles can be found at [US Census TIGER/Line Shapefiles](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html)

In [ ]:
import geopandas as gpd
import numpy as np

In [ ]:
lon_min, lon_max = -100, -90
lat_min, lat_max = 25, 35

In [ ]:
county = gpd.read_file("shp/tl_2025_us_county.shp").cx[lon_min:lon_max, lat_min:lat_max]
state = gpd.read_file("shp/tl_2025_us_state.shp").cx[lon_min:lon_max, lat_min:lat_max]
county.geometry = county.geometry.boundary
state.geometry = state.geometry.boundary
county = county.dissolve()
state = state.dissolve()
county_minus_state = county.overlay(state, how="difference")

In [ ]:
layers = [state, county_minus_state]
names  = ['State', 'County']
verts = []
total = 0
for name, layer in zip(names, layers):
    geom = layer.iloc[0].geometry
    lines = geom.geoms if hasattr(geom, 'geoms') else [geom]
    for line in lines:
        coords = np.array(line.coords)
        for i in range(len(coords) - 1):
            verts.extend([*coords[i], *coords[i+1]])
            total += 1
    print(f"{name}: sources -> {total*2}")
np.array(verts, dtype=np.float32).tofile('map.bin')